In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('./data/accepted_2007_to_2018Q4.csv')

df.head()

C:\Users\vecto\AppData\Local\Temp\ipykernel_9448\2214727301.py:4: DtypeWarning: Columns (0: id, 1: desc, 2: next_pymnt_d, 3: verification_status_joint, 4: sec_app_earliest_cr_line, 5: hardship_type, 6: hardship_reason, 7: hardship_status, 8: hardship_start_date, 9: hardship_end_date, 10: payment_plan_start_date, 11: hardship_loan_status, 12: debt_settlement_flag_date, 13: settlement_status, 14: settlement_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./data/accepted_2007_to_2018Q4.csv')


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


Look at null percentages for each column:

In [9]:
df.isna().sum().sort_values(ascending=False)/len(df)*100

member_id                                     100.000000
orig_projected_additional_accrued_interest     99.617331
hardship_reason                                99.517097
hardship_payoff_balance_amount                 99.517097
hardship_last_payment_amount                   99.517097
                                                 ...    
total_rec_int                                   0.001460
disbursement_method                             0.001460
hardship_flag                                   0.001460
debt_settlement_flag                            0.001460
id                                              0.000000
Length: 151, dtype: float64

In [8]:
df.shape

(2260701, 151)

Drop columns that could not exist without issuing a loan:

In [10]:
cols_to_drop = [
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',
    'out_prncp', 'out_prncp_inv', 'loan_status', 'funded_amnt', 'funded_amnt_inv',
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date',
    'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status',
    'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount',
    'hardship_last_payment_amount', 'debt_settlement_flag', 'debt_settlement_flag_date',
    'settlement_status', 'settlement_date', 'settlement_amount', 'settlement_percentage',
    'settlement_term', 'collections_12_mths_ex_med', 'policy_code', 'url', 'desc',
    'title', 'zip_code', 'id', 'member_id', 'sub_grade'
]

df = df.drop(columns=cols_to_drop)

Dates:

In [11]:
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')

Convert to months:

In [12]:
df['credit_history_months'] = (
    (df['issue_d'].dt.year - df['earliest_cr_line'].dt.year) * 12 +
    (df['issue_d'].dt.month - df['earliest_cr_line'].dt.month)
)
df = df.drop(columns=['earliest_cr_line'])

Convert months to integers:

In [ ]:
df = df.dropna(subset=['term']) # Drop rows with missing 'term' values
df['term'] = df['term'].str.strip().str.replace(' months', '').astype(int)

Encode employment length:

In [16]:
emp_length_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
    '10+ years': 10
}
df['emp_length'] = df['emp_length'].map(emp_length_map)

Handle categorical variables:

In [20]:
categorical_cols = ['purpose', 'home_ownership', 'verification_status']
for col in categorical_cols:
    df[col] = df[col].astype('category')

In [ ]:
pd.to_csv(df, './data/cleaned_data.csv', index=False)

Summary:

- Engineer dates into credit history length
- Encode term and employment length (emp_length)
- Update categorical variables to 'category' type
- Don't need to drop all NA values since our model can handle them